In [0]:
data_emp = [
    (101, "Rahul", 10, 75000,'01-03-2023' ,"rahul@gmail.com"),
    (102, "Sneha", 10, 82000,'15-05-2019',"sneha@gamil.com"),
    (103, "Amit", 10, 82000,'07-07-2024',"amit123@outlook.com"),
    (104, "Neha", 20, 65000,'17-12-2025',"neha23@gmail.com"),
    (105, "Kiran", 20, 70000,'26-04-2022',"kiran28@gmail.com"),
    (106, "Pooja", 20, 90000,'11-09-2023',"pooja65@gmail.com"),
    (107, "Ravi", 30, 60000,'18-08-2021',"ravi@34gmail.com"),
    (108, "Anita", 30, 64000,'12-04-2026',"anita@gmail.com"),
    (109, "Vikas", 30, 72000,'02-09-2018',"vikas12@gmail.com"),
    (110, "Meena", 40, 50000,'11-07-2021',"meena211@gamil.com")
]

columns = ["emp_id", "name", "dept_id", "salary", "hire_date", "email"]

emp_df = spark.createDataFrame(data_emp, columns)

emp_df.display()

emp_id,name,dept_id,salary,hire_date,email
101,Rahul,10,75000,01-03-2023,rahul@gmail.com
102,Sneha,10,82000,15-05-2019,sneha@gamil.com
103,Amit,10,82000,07-07-2024,amit123@outlook.com
104,Neha,20,65000,17-12-2025,neha23@gmail.com
105,Kiran,20,70000,26-04-2022,kiran28@gmail.com
106,Pooja,20,90000,11-09-2023,pooja65@gmail.com
107,Ravi,30,60000,18-08-2021,ravi@34gmail.com
108,Anita,30,64000,12-04-2026,anita@gmail.com
109,Vikas,30,72000,02-09-2018,vikas12@gmail.com
110,Meena,40,50000,11-07-2021,meena211@gamil.com


In [0]:
data_dept = [
    (10, "IT"),
    (20, "HR"),
    (30, "Finance"),
    (40, "Admin"),
    (50, "Sales")   # no employees (for anti join practice)
]

columns = ["dept_id", "dept_name"]

dept_df = spark.createDataFrame(data_dept, columns)

dept_df.display()


dept_id,dept_name
10,IT
20,HR
30,Finance
40,Admin
50,Sales


In [0]:
# 1. find top 2 highest paid employees in each department .


from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("rn", row_number().over(w)) \
    .filter(col("rn") <= 2) \
    .select("name", "dept_id", "salary")

df.display()



name,dept_id,salary
Sneha,10,82000
Amit,10,82000
Pooja,20,90000
Kiran,20,70000
Vikas,30,72000
Anita,30,64000
Meena,40,50000


In [0]:
# 2. find employees whose salary is greater than the average salary of their department.

from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id")

df= emp_df.withColumn("avg_sal",avg(col("salary")).over(w))\
    .filter(col("salary") > col("avg_sal"))\
    .select("name","dept_id","salary","avg_sal")
df.display()        

name,dept_id,salary,avg_sal
Sneha,10,82000,79666.66666666667
Amit,10,82000,79666.66666666667
Pooja,20,90000,75000.0
Vikas,30,72000,65333.333333333336


In [0]:
# 3. Find 2nd highest salary per department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df= emp_df.withColumn("sec_highest_sal",row_number().over(w))\
    .filter(col("sec_highest_sal") == 2)\
    .select("name","dept_id", "salary", "sec_highest_sal")
df.display()        

name,dept_id,salary,sec_highest_sal
Amit,10,82000,2
Kiran,20,70000,2
Anita,30,64000,2


In [0]:
# 4. Find employees not mapped to any department. 

from pyspark.sql.functions import *

df = emp_df.join(dept_df, on ="dept_id", how = "left_anti")\
    
df.display()    


dept_id,emp_id,name,salary


In [0]:
# 5. Find department with highest average salary.

from pyspark.sql.functions import *


df = emp_df.groupBy("dept_id") \
           .agg(avg("salary").alias("avg_salary")) \
           .orderBy(col("avg_salary").desc()) \
           .limit(1)

df.display()




dept_id,avg_salary
10,79666.66666666667


In [0]:
# 6. Find department where total salary is highest.

from pyspark.sql.functions import *
df = emp_df.groupBy("dept_id").agg(sum("salary").alias("total_sal")).orderBy(col("total_sal").desc())\
    .limit(1)\
    .select("dept_id","total_sal")    
df.display()    

dept_id,total_sal
10,239000


In [0]:
# 3. Find employees whose salary is greater than both previous AND next employee within their department.

from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df.withColumn("prev_sal",lag("salary").over(w))\
    .withColumn("next_sal",lead("salary").over(w))\
    .filter(
        col("prev_sal").isNull() & 
        col("next_sal").isNull()&
        (col("salary") > col("prev_sal")) & 
        (col("salary") > col("next_sal"))
        )\
    .select("name","dept_id")  

df.display()          



name,dept_id


In [0]:
# 2. Find top 2 highest paid employees in each department with department name.

from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.join(dept_df, on="dept_id", how="left") \
    .withColumn("rn", row_number().over(w)) \
    .filter(col("rn") <= 2) \
    .select("name", "dept_name", "salary", "rn")

df.display()

name,dept_name,salary,rn
Amit,IT,82000,1
Sneha,IT,82000,2
Pooja,HR,90000,1
Kiran,HR,70000,2
Vikas,Finance,72000,1
Anita,Finance,64000,2
Meena,Admin,50000,1


In [0]:
# 3. Find employees whose salary is greater than both previous AND next employee within their department.
df = emp_df.withColumn("prev_sal", lag("salary").over(w)) \
    .withColumn("next_sal", lead("salary").over(w)) \
    .filter(
        col("prev_sal").isNotNull() &
        col("next_sal").isNotNull() &
        (col("salary") > col("prev_sal")) &
        (col("salary") > col("next_sal"))
    ) \
    .select("name", "dept_id")

df.display()

name,dept_id


In [0]:
###################  IMP Questions. ############################

# 1. Find employees: working in IT department and salary greater than 70000.

     

df = emp_df.join(dept_df, "dept_id") \
    .filter((col("salary") > 70000) & (col("dept_name") == "IT")) \
    .select("name", "salary", "dept_name")
df.display()

name,salary,dept_name
Sneha,82000,IT
Rahul,75000,IT
Amit,82000,IT


In [0]:
# 2. total salary paid per department.

from pyspark.sql.functions import *
df= emp_df.groupBy("dept_id").agg(sum("salary").alias("total_sal")).orderBy(col("dept_id"))\
    .select("dept_id","total_sal")
df.display()    

dept_id,total_sal
10,239000
20,225000
30,196000
40,50000


In [0]:
# 3. Find employees who: do NOT belong to any department .

from pyspark.sql.functions import *

df = emp_df.join(dept_df, on = "dept_id", how = "left_anti")\
    .select("name","dept_id")
df.display()    

name,dept_id


In [0]:
# 4. highest paid employee in each department
from pyspark.sql.functions import *
from pyspark.sql.window import Window

w  = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("rn", row_number().over(w))\
    .filter(col("rn") ==  1)\
    .select("name", "salary","rn", "dept_id")
df.display()    




name,salary,rn,dept_id
Sneha,82000,1,10
Pooja,90000,1,20
Vikas,72000,1,30
Meena,50000,1,40


In [0]:
# 5. salary is greater than average salary of their department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id")
df = emp_df.withColumn("avg_sal", avg("salary").over(w))\
    .filter(col("salary") > col("avg_sal"))\
    .select("name","salary","avg_sal")
df.display()    


name,salary,avg_sal
Sneha,82000,79666.66666666667
Amit,82000,79666.66666666667
Pooja,90000,75000.0
Vikas,72000,65333.333333333336


In [0]:
# 6.  Find: top 3 highest salaries in each department.
from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("rn",dense_rank().over(w))\
    .filter(col("rn") <= 3)\
    .select("name","dept_id","salary","rn")
df.display()    

name,dept_id,salary,rn
Sneha,10,82000,1
Amit,10,82000,1
Rahul,10,75000,2
Pooja,20,90000,1
Kiran,20,70000,2
Neha,20,65000,3
Vikas,30,72000,1
Anita,30,64000,2
Ravi,30,60000,3
Meena,40,50000,1


In [0]:
# 7. Find: department name with employee count.

from pyspark.sql.functions import *
df = emp_df.join(dept_df, on ="dept_id").groupBy("dept_name").agg((count("*").alias("emp_count")))\
    .select("dept_name","emp_count")
df.display()    
    

dept_name,emp_count
HR,3
Finance,3
Admin,1
IT,3


In [0]:
# 8. Find departments: having more than 2 employees.

from pyspark.sql.functions import *

df = emp_df.join(dept_df, on = "dept_id").groupBy("dept_name").agg((count("*").alias("emp_count")))\
    .filter(col("emp_count") >2)\
    .select("dept_name","emp_count")
df.display()        

dept_name,emp_count
HR,3
Finance,3
IT,3


In [0]:
# 9. salary is greater than previous employee salary within department.

from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id").orderBy(col("salary"))
df = emp_df.withColumn("prev_sal",lag("salary").over(w))\
    .filter( col("salary") > col("prev_sal"))

   
df.display()    

emp_id,name,dept_id,salary,prev_sal
102,Sneha,10,82000,75000
105,Kiran,20,70000,65000
106,Pooja,20,90000,70000
108,Anita,30,64000,60000
109,Vikas,30,72000,64000


In [0]:
# 10. find second highest paid employee in each department with department name.
from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_name").orderBy(col("salary").desc())

df = emp_df.join(dept_df, on = "dept_id").withColumn("rn",dense_rank().over(w))\
    .filter(col("rn") == 2)\
    .select("name","salary","dept_name")   
df.display()     

name,salary,dept_name
Anita,64000,Finance
Kiran,70000,HR
Rahul,75000,IT


In [0]:
# 11. Find employees who are in top 3 highest salaries in each department (include duplicates).

from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("top_sal",dense_rank().over(w))\
    .filter(col("top_sal") <= 3)\
     .select("name","dept_id","top_sal")
df.display()        

name,dept_id,top_sal
Amit,10,1
Sneha,10,1
Rahul,10,2
Pooja,20,1
Kiran,20,2
Neha,20,3
Vikas,30,1
Anita,30,2
Ravi,30,3
Meena,40,1


In [0]:
# 12. Calculate running total of salary within each department ordered by salary.
from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df.withColumn("running_total", sum("salary").over(w))\
    .select("name","dept_id","salary","running_total")
df.display()    

name,dept_id,salary,running_total
Rahul,10,75000,75000
Sneha,10,82000,239000
Amit,10,82000,239000
Neha,20,65000,65000
Kiran,20,70000,135000
Pooja,20,90000,225000
Ravi,30,60000,60000
Anita,30,64000,124000
Vikas,30,72000,196000
Meena,40,50000,50000


In [0]:
# 13. Find employees where the difference between their salary and previous employee salary is greater than 10000 (within department).

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df.withColumn("prev_emp_sal",lag("salary").over(w))
result = df.filter((col("salary") - col("prev_emp_sal")) > 10000)
result.display()

emp_id,name,dept_id,salary,prev_emp_sal
106,Pooja,20,90000,70000


In [0]:
# 14. Find top 2 departments having highest total salary.

from pyspark.sql.functions import *


df = emp_df.groupBy("dept_id")\
    .agg(sum("salary").alias("highest_total_sal")).orderBy(col("highest_total_sal").desc())\
    .limit(2)
    
df.display()


dept_id,highest_total_sal
10,239000
20,225000


In [0]:
# 15.  Find employees who belong to departments where average salary is greater than 70000.

from pyspark.sql.functions import *


dept_avg = emp_df.groupBy("dept_id") \
    .agg(avg("salary").alias("avg_salary")) \
    .filter(col("avg_salary") > 70000)
df.display()


dept_id,emp_id,name,salary,avg_salary
10,101,Rahul,75000,79666.66666666667
10,102,Sneha,82000,79666.66666666667
10,103,Amit,82000,79666.66666666667
20,104,Neha,65000,75000.0
20,105,Kiran,70000,75000.0
20,106,Pooja,90000,75000.0


In [0]:
# 16. Find top 2 highest paid employees in each department along with department name.

from pyspark.sql.functions import *

from pyspark.sql.window import Window

w = Window.partitionBy("dept_name").orderBy(col("salary").desc())

df = emp_df.join(dept_df, on = "dept_id")\
    .withColumn("rn",dense_rank().over(w))\
    .filter(col("rn") <= 2)\
    .select("name","dept_name","rn")      
df.display()    


name,dept_name,rn
Meena,Admin,1
Vikas,Finance,1
Anita,Finance,2
Pooja,HR,1
Kiran,HR,2
Sneha,IT,1
Amit,IT,1
Rahul,IT,2


In [0]:
# 17. Find employees whose salary is less than the next employee salary (within department).
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df.withColumn("next_sal",lead("salary").over(w))\
    .filter(col("salary") < col("next_sal"))\
    .select("name","salary","next_sal")
df.display()        

name,salary,next_sal
Rahul,75000,82000
Neha,65000,70000
Kiran,70000,90000
Ravi,60000,64000
Anita,64000,72000


In [0]:
# 18. Find departments where all employees have salary greater than 60000.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

  

df = emp_df.groupBy("dept_id") \
    .agg(min("salary").alias("min_salary")) \
    .filter(col("min_salary") > 60000) \
    .select("dept_id")

df.display()

dept_id
10
20


In [0]:
# 19.  Find employees who earn more than the highest salary of another department.
from pyspark.sql.functions import max, col

# Step 1: max salary per department
max_sal_df = emp_df.groupBy("dept_id") \
    .agg(max("salary").alias("max_salary"))

# Step 2: join and filter
df = emp_df.alias("e") \
    .join(max_sal_df.alias("m"), col("e.dept_id") != col("m.dept_id")) \
    .filter(col("e.salary") > col("m.max_salary")) \
    .select("e.name", "e.dept_id", "e.salary") \
    .distinct()

df.display()


name,dept_id,salary
Rahul,10,75000
Amit,10,82000
Pooja,20,90000
Anita,30,64000
Sneha,10,82000
Neha,20,65000
Kiran,20,70000
Ravi,30,60000
Vikas,30,72000


In [0]:
# 20. Find employees whose salary is the Nth highest (N = 3) in their department. 
from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("nth_sal",dense_rank().over(w))\
    .filter(col("nth_sal") == 3 )\
    .select("name","dept_id","nth_sal")
df.display()        



name,dept_id,nth_sal
Neha,20,3
Ravi,30,3


In [0]:
# 21. Find employees whose salary is greater than 70000.

from pyspark.sql.functions import *

df = emp_df.filter(col("salary") > 70000).select("name","salary")

df.display()


name,salary
Rahul,75000
Sneha,82000
Amit,82000
Pooja,90000
Vikas,72000


In [0]:
# 22. Find employees: working in IT department and salary greater than 70000 .

from pyspark.sql.functions import *

df = emp_df.join(dept_df, on = "dept_id").filter((col("dept_name") == "IT") & (col("salary") > 70000))\
    .select("name", "dept_name","salary")
df.display()        

name,dept_name,salary
Rahul,IT,75000
Sneha,IT,82000
Amit,IT,82000


In [0]:
# 23. Find employees whose salary is between 60000 and 80000.

from pyspark.sql.functions import *

df = emp_df.filter((col("salary") >= 60000) & (col("salary") <= 80000))\
    .select("name","salary")
df.display()    


name,salary
Rahul,75000
Neha,65000
Kiran,70000
Ravi,60000
Anita,64000
Vikas,72000


In [0]:
# 24.Find employees who are not working in HR department.

from pyspark.sql.functions import *

df = emp_df.join(dept_df, on = "dept_id")\
    .filter(col("dept_name") != "HR")\
    .select("name","dept_name")    
df.display()

name,dept_name
Rahul,IT
Amit,IT
Sneha,IT
Anita,Finance
Ravi,Finance
Vikas,Finance
Meena,Admin


In [0]:
# 25. Find employees who satisfy: salary > 60000 department = Finance OR IT.

from pyspark.sql.functions import *

df = emp_df.join(dept_df, on ="dept_id")\
    .filter((col("salary") > 60000) & ((col("dept_name") == "Finance") |  (col("dept_name") ==  "IT")))\
    .select("name","salary","dept_name")
df.display()    

name,salary,dept_name
Sneha,82000,IT
Rahul,75000,IT
Amit,82000,IT
Vikas,72000,Finance
Anita,64000,Finance


In [0]:
# 26.  Find the total salary paid in each department.
from pyspark.sql.functions import *

df = emp_df.groupBy("dept_id")\
    .agg(sum("salary").alias("total"))\
    .select("total","dept_id")    
df.display()    

total,dept_id
239000,10
225000,20
196000,30
50000,40


In [0]:
# 27. Find the average salary in each department.

from pyspark.sql.functions import *

df = emp_df.groupBy("dept_id").agg(avg("salary").alias("avg_salary"))\
    .select("dept_id","avg_salary")
df.display()    

dept_id,avg_salary
10,79666.66666666667
20,75000.0
30,65333.333333333336
40,50000.0


In [0]:
# 28. Find the number of employees in each department.

df = emp_df.groupBy("dept_id").agg(count("*").alias("no_of_emp"))\
    .select("dept_id","no_of_emp")
df.display()    


dept_id,no_of_emp
10,3
20,3
30,3
40,1


In [0]:
# 29.Find departments having more than 2 employees.

from pyspark.sql.functions import *

df = emp_df.groupBy("dept_id").agg(count("*").alias("emp_count"))\
    .filter(col("emp_count") > 2)\
     .select("dept_id","emp_count")
df.display()        

dept_id,emp_count
10,3
20,3
30,3


In [0]:
# 30.find the department with the highest total salary.

from pyspark.sql.functions import *

df = emp_df.groupBy("dept_id")\
    .agg(sum("salary").alias("total_sal"))\
    .orderBy(col("total_sal").desc()).limit(1)\
    .select("dept_id","total_sal")
df.display()        


dept_id,total_sal
10,239000


In [0]:
# 31.  Find employee name along with department name.
from pyspark.sql.functions import*

df = emp_df.join(dept_df, on = "dept_id", how = "inner")\
    .select("name","dept_name")
df.display()    

name,dept_name
Rahul,IT
Sneha,IT
Amit,IT
Neha,HR
Kiran,HR
Pooja,HR
Ravi,Finance
Anita,Finance
Vikas,Finance
Meena,Admin


In [0]:
# 32.  Find all employees and their department names.

from pyspark.sql.functions import *
df = emp_df.join(dept_df, on = "dept_id", how = "left")\
    .select("name","dept_name")
df.display()    

name,dept_name
Rahul,IT
Sneha,IT
Amit,IT
Neha,HR
Kiran,HR
Pooja,HR
Ravi,Finance
Anita,Finance
Vikas,Finance
Meena,Admin


In [0]:
# 33. Find employees who do NOT belong to any department.
from pyspark.sql.functions import *

df = emp_df.join(dept_df, on = "dept_id", how = "left_anti")\
    .select("name", "dept_id")
df.display()    



name,dept_id


In [0]:
# 34. Find departments that do NOT have any employees.

from pyspark.sql.functions import *

df = dept_df.join(emp_df, on="dept_id", how="left_anti") \
    .select("dept_id", "dept_name")

df.display()



dept_id,dept_name
50,Sales


In [0]:
# 35. Find employee count for each department along with department name.

from pyspark.sql.functions import *

df = emp_df.join(dept_df, on = "dept_id", how = "left").groupBy("dept_name")\
    .agg(count("*").alias("emp_count"))\
    .select("dept_name","emp_count")
df.display()        

dept_name,emp_count
IT,3
HR,3
Finance,3
Admin,1


In [0]:
#  Scenario : You have two tables: emp_df, dept_df
# Business Requirement
#   Management wants a report showing:
#   Department name
#   Total employees
#   Average salary
#   Only departments having more than 2 employees
#   Sort by average salary descending

from pyspark.sql.functions import *

df = emp_df.join(dept_df, on = "dept_id")\
    .groupBy("dept_name")\
    .agg(count("*").alias("Total_emp"), avg("salary").alias("avg_salary"))\
    .filter(col("Total_emp") > 2)\
    .orderBy(col("avg_salary").desc()).select("dept_name","Total_emp","avg_salary")    
df.display()        



dept_name,Total_emp,avg_salary
IT,3,79666.66666666667
HR,3,75000.0
Finance,3,65333.333333333336


In [0]:
# 36. Find highest paid employee in each department.

from pyspark.sql.functions import *

from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("highest_emp",dense_rank().over(w))\
    .filter(col("highest_emp") == 1)\
    .select("name","salary","dept_id")   
df.display()    


name,salary,dept_id
Amit,82000,10
Sneha,82000,10
Pooja,90000,20
Vikas,72000,30
Meena,50000,40


In [0]:
# 37. Find top 2 highest paid employees in each department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("rnk",dense_rank().over(w))\
    .filter(col("rnk") <= 2)\
    .select("name","dept_id","salary")
df.display()         

name,dept_id,salary
Sneha,10,82000
Amit,10,82000
Rahul,10,75000
Pooja,20,90000
Kiran,20,70000
Vikas,30,72000
Anita,30,64000
Meena,40,50000


In [0]:
# 38. Find 2nd highest salary in each department.
from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())
   
df = emp_df.withColumn("rnk",dense_rank().over(w))\
    .filter(col("rnk") == 2)\
    .select("salary","dept_id")
df.display()           


salary,dept_id
75000,10
70000,20
64000,30


In [0]:
# 39. Find top 3 salaries including duplicates.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("rnk",dense_rank().over(w))\
    .filter(col("rnk") <=3)\
    .select("rnk","salary")
df.display()    

rnk,salary
1,82000
1,82000
2,75000
1,90000
2,70000
3,65000
1,72000
2,64000
3,60000
1,50000


In [0]:
# 40. Rank employees by salary within department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary").desc())

df = emp_df.withColumn("rank",dense_rank().over(w))\
    .select("rank","name","salary")
df.display()    


rank,name,salary
1,Amit,82000
1,Sneha,82000
2,Rahul,75000
1,Pooja,90000
2,Kiran,70000
3,Neha,65000
1,Vikas,72000
2,Anita,64000
3,Ravi,60000
1,Meena,50000


In [0]:
 # senario 2 :  Generate a department leaderboard showing top earners.


from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, dense_rank

# Highest paid employee per department
w1 = Window.partitionBy("dept_id").orderBy(col("salary").desc())

top_emp = emp_df.withColumn("rn", row_number().over(w1)) \
    .filter(col("rn") == 1)

# Join department names
top_emp_dept = top_emp.join(dept_df, on="dept_id")

# Department leaderboard
w2 = Window.orderBy(col("salary").desc())

leaderboard = top_emp_dept.withColumn(
    "dept_rank",
    dense_rank().over(w2)
).select(
    "dept_name",
    "name",
    "salary",
    "dept_rank"
)

leaderboard.display()
 

dept_name,name,salary,dept_rank
HR,Pooja,90000,1
IT,Amit,82000,2
Finance,Vikas,72000,3
Admin,Meena,50000,4


In [0]:
# 41. Find employees whose salary is greater than the previous employee salary within the same department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df.withColumn("prev_sal",lag("salary").over(w))\
    .filter(col("salary") > col("prev_sal"))\
    .select("name","salary", "prev_sal","dept_id")
df.display()        

name,salary,prev_sal,dept_id
Sneha,82000,75000,10
Kiran,70000,65000,20
Pooja,90000,70000,20
Anita,64000,60000,30
Vikas,72000,64000,30


In [0]:
# 42. Find employees whose salary is less than the next employee salary within the same department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df.withColumn("next_sal",lead("salary").over(w))\
    .filter(col("salary") < col("next_sal"))\
     .select("name","salary","next_sal","dept_id")
df.display()        

name,salary,next_sal,dept_id
Rahul,75000,82000,10
Neha,65000,70000,20
Kiran,70000,90000,20
Ravi,60000,64000,30
Anita,64000,72000,30


In [0]:
# 43. Find employees where the difference between current salary and previous salary is greater than 10000.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df\
    .withColumn("prev_sal",lag("salary").over(w))\
    .withColumn("salary_diff", col("salary") - col("prev_sal"))\
    .filter(col("salary_diff") > 10000)\
    .select("name","salary","prev_sal","salary_diff")     
df.display()       

name,salary,prev_sal,salary_diff
Pooja,90000,70000,20000


In [0]:
#44. Find employees whose salary is greater than both previous and next employee salary within the department.

from  pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df\
    .withColumn("prev_sal", lag("salary").over(w))\
    .withColumn("next_sal",lead("salary").over(w))\
    .filter(col("prev_sal").isNotNull() &
            col("next_sal").isNotNull() & 
            (col("salary") > col("prev_sal"))  &  (col("salary") > col("next_sal"))
        )\
    .select("name","salary")
df.display()


name,salary


In [0]:
# 45.Find employees whose salary is less than both previous and next employee salary within the department.



from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df \
    .withColumn("prev_sal", lag("salary").over(w)) \
    .withColumn("next_sal", lead("salary").over(w)) \
    .filter(
        col("prev_sal").isNotNull() &
        col("next_sal").isNotNull() &
        (col("salary") < col("prev_sal")) &
        (col("salary") < col("next_sal"))
    ) \
    .select("name", "dept_id", "salary", "prev_sal", "next_sal")

df.display()

name,dept_id,salary,prev_sal,next_sal


In [0]:
# Scenario 3 : 
#   HR wants to identify employees who received a significant salary jump.
#   Find employees where:

#   Current salary is greater than previous employee salary
#   Salary difference is greater than 15000
#   Show department name also


from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_name").orderBy(col("salary"))

df = emp_df.join(dept_df, on = "dept_id")\
    .withColumn("prev_sal", lag("salary").over(w))\
    .withColumn("salary_diff", col("salary") - col("prev_sal"))\
    .filter((col("salary") > col("prev_sal")) & (col("salary_diff") > 15000))\
    .select("name","salary_diff","salary","prev_sal","dept_name")    
df.display()        

name,salary_diff,salary,prev_sal,dept_name
Pooja,20000,90000,70000,HR


In [0]:
# 46. Find employees whose salary is greater than the average salary of their department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df.withColumn("avg_sal",avg("salary").over(w))\
    .filter(col("salary") > col("avg_sal"))\
    .select("name","dept_id","salary","avg_sal")    
df.display()

name,dept_id,salary,avg_sal
Amit,10,82000,79666.66666666667
Sneha,10,82000,79666.66666666667
Kiran,20,70000,67500.0
Pooja,20,90000,75000.0
Anita,30,64000,62000.0
Vikas,30,72000,65333.333333333336


In [0]:
# 47.Find employees whose salary is greater than the overall company average salary.

from pyspark.sql.functions import *

company_avg_df = emp_df.agg(avg("salary").alias("company_avg"))
df = emp_df.crossJoin(company_avg_df)\
    .filter(col("salary") > col("company_avg"))\
    .select("name","salary","company_avg")
df.display()

name,salary,company_avg
Rahul,75000,71000.0
Sneha,82000,71000.0
Amit,82000,71000.0
Pooja,90000,71000.0
Vikas,72000,71000.0


In [0]:
# 48. Find departments whose average salary is greater than the company average salary.

from pyspark.sql.functions import *

# Company average
comp_avg_df = emp_df.agg(
    avg("salary").alias("company_avg")
)

# Department averages
dept_avg_df = emp_df.groupBy("dept_id") \
    .agg(avg("salary").alias("dept_avg"))

# Compare department avg with company avg
df = dept_avg_df.crossJoin(comp_avg_df) \
    .filter(col("dept_avg") > col("company_avg")) \
    .select("dept_id", "dept_avg", "company_avg")

df.display()       




dept_id,dept_avg,company_avg
10,79666.66666666667,71000.0
20,75000.0,71000.0


In [0]:
# 49. Find employees whose salary is greater than: department average salary AND company average salary.


from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Department average
w = Window.partitionBy("dept_id")

# Company average
company_avg_df = emp_df.agg(
    avg("salary").alias("company_avg")
)

df = emp_df \
    .withColumn(
        "dept_avg",
        avg("salary").over(w)
    ) \
    .crossJoin(company_avg_df) \
    .filter(
        (col("salary") > col("dept_avg")) &
        (col("salary") > col("company_avg"))
    ) \
    .select(
        "name",
        "dept_id",
        "salary",
        "dept_avg",
        "company_avg"
    )

df.display()



name,dept_id,salary,dept_avg,company_avg
Sneha,10,82000,79666.66666666667,71000.0
Amit,10,82000,79666.66666666667,71000.0
Pooja,20,90000,75000.0,71000.0
Vikas,30,72000,65333.333333333336,71000.0


In [0]:

# 50. Find the department with the highest average salary.

from pyspark.sql.functions import *

df = emp_df.groupBy("dept_id") \
    .agg(avg("salary").alias("avg_salary")) \
    .orderBy(col("avg_salary").desc()) \
    .limit(1)

df.display()




dept_id,avg_salary
10,79666.66666666667


In [0]:
# Senario 4:  Business Requirement := Management wants to identify star employees.
# Find employees who:
# Earn more than their department average salary
# Earn more than the company average salary
# Belong to departments having more than 2 employees
# Show: name , dept_name,  salary, dept_avg, company_avg.

from pyspark.sql.functions import *




In [0]:
# 51. Find employees whose salary is: greater than their department average salary and belong to departments having more than 2 employees.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# calculate dept avg.
w = Window.partitionBy("dept_id")

dept_avg_df = emp_df.withColumn("dept_avg",avg("salary").over(w))

# calulate emp count.

emp_count_df = emp_df.groupBy("dept_id")\
    .agg(count("*").alias("emp_count"))

df = dept_avg_df.join(emp_count_df, on = "dept_id")\
    .filter((col("salary") > col("dept_avg")) & (col("emp_count") > 2))\
    .select(
        "name",
        "dept_id",
        "salary",
        "dept_avg",
        "emp_count"
    )
        
df.display()    

name,dept_id,salary,dept_avg,emp_count
Sneha,10,82000,79666.66666666667,3
Amit,10,82000,79666.66666666667,3
Pooja,20,90000,75000.0,3
Vikas,30,72000,65333.333333333336,3


In [0]:
# 52. Find employees whose salary is greater than the company average salary.
from pyspark.sql.functions import *




comp_avg_df = emp_df.agg(avg("salary").alias("comp_avg"))

df = emp_df.crossJoin(comp_avg_df)\
    .filter(col("salary") > col("comp_avg"))\
    .select("name","salary","comp_avg")

df.display()    



name,salary,comp_avg
Rahul,75000,71000.0
Sneha,82000,71000.0
Amit,82000,71000.0
Pooja,90000,71000.0
Vikas,72000,71000.0


In [0]:
# 53 . Find employees who: salary > department average AND salary > company average.

 
from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id")

# Department average on each employee row
emp_with_dept_avg = emp_df.withColumn(
    "dept_avg",
    avg("salary").over(w)
)

# Company average
comp_avg_df = emp_df.agg(
    avg("salary").alias("comp_avg")
)

# Compare against both
df = emp_with_dept_avg.crossJoin(comp_avg_df) \
    .filter(
        (col("salary") > col("dept_avg")) &
        (col("salary") > col("comp_avg"))
    ) \
    .select(
        "name",
        "dept_id",
        "salary",
        "dept_avg",
        "comp_avg"
    )

df.display() 

name,dept_id,salary,dept_avg,comp_avg
Sneha,10,82000,79666.66666666667,71000.0
Amit,10,82000,79666.66666666667,71000.0
Pooja,20,90000,75000.0,71000.0
Vikas,30,72000,65333.333333333336,71000.0


In [0]:
# 54. Find employees whose salary is: less than their department average salary but greater than the company average salary 

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id")

df_dept_a = emp_df.withColumn("dpt_average", avg("salary").over(w))

df_comp_a = emp_df.agg(avg("salary").alias("comp_average"))

df = df_dept_a.crossJoin(df_comp_a)\
    .filter((col("salary") < col("dpt_average")) & (col("salary") > col("comp_average")))\
        .select("name","dept_id","salary","dpt_average","comp_average")
df.display()        

name,dept_id,salary,dpt_average,comp_average
Rahul,10,75000,79666.66666666667,71000.0


In [0]:
# 55. Find departments whose:average salary is greater than company average salary and employee count is greater than 2.

from pyspark.sql.functions import *


df1 = emp_df.groupBy("dept_id").agg(avg("salary").alias("d_avg"), count("*").alias("emp_count"))

df2 = emp_df.agg(avg("salary").alias("c_avg"))



df = df1.crossJoin(df2)\
    .filter((col("d_avg") > col("c_avg")) & (col("emp_count") > 2))\
        .select("dept_id","d_avg","c_avg","emp_count")
df.display()        

dept_id,d_avg,c_avg,emp_count
10,79666.66666666667,71000.0,3
20,75000.0,71000.0,3


In [0]:
# 56 Find employees whose salary is:greater than the department average salary but less than the highest salary in their department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window
w= Window.partitionBy("dept_id")

df = emp_df.withColumn("dept_avg",avg("salary").over(w))\
    .withColumn("dept_high_sal",max("salary").over(w))\
        .filter((col("salary") > col("dept_avg")) & (col("salary") < col("dept_high_sal")) )\
    .select("name","salary","dept_avg","dept_high_sal")    
df.display()

name,salary,dept_avg,dept_high_sal


In [0]:
# 57. Find departments where:average salary is greater than company average salary total salary is greater than    200000

from pyspark.sql.functions import *

df_c = emp_df.agg(avg("salary").alias("c_avg"), )

df_avg = emp_df.groupBy("dept_id").agg(avg("salary").alias("d_avg"),sum("salary").alias("total"))

df = df_c.crossJoin(df_avg)\
    .filter((col("d_avg") > col("c_avg")) & (col("total") > 200000))\
        .select("d_avg","c_avg","total")


df.display()


d_avg,c_avg,total
79666.66666666667,71000.0,239000
75000.0,71000.0,225000


In [0]:
# 58. Find employees who:are NOT the highest-paid employee in their department earn more than department average salary.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id")

df = emp_df \
    .withColumn("dept_avg", avg("salary").over(w)) \
    .withColumn("dept_max", max("salary").over(w)) \
    .filter(
        (col("salary") > col("dept_avg")) &
        (col("salary") < col("dept_max"))
    ) \
    .select(
        "name",
        "dept_id",
        "salary",
        "dept_avg",
        "dept_max"
    )

df.display()

name,dept_id,salary,dept_avg,dept_max


In [0]:
# 60.   Management wants a High Potential Employee Report.
#Find employees who:

#are in Top 3 salaries of their department
#salary > department average salary
#salary > company average salary
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Ranking window
w_rank = Window.partitionBy("dept_id") \
               .orderBy(col("salary").desc())

# Department average window
w_avg = Window.partitionBy("dept_id")

# Company average
company_avg_df = emp_df.agg(
    avg("salary").alias("company_avg")
)

df = emp_df \
    .withColumn(
        "rank",
        dense_rank().over(w_rank)
    ) \
    .withColumn(
        "dept_avg",
        avg("salary").over(w_avg)
    ) \
    .crossJoin(company_avg_df) \
    .join(dept_df, "dept_id") \
    .filter(
        (col("rank") <= 3) &
        (col("salary") > col("dept_avg")) &
        (col("salary") > col("company_avg"))
    ) \
    .select(
        "name",
        "dept_name",
        "salary",
        "rank",
        "dept_avg",
        "company_avg"
    )

df.display()

name,dept_name,salary,rank,dept_avg,company_avg
Sneha,IT,82000,1,79666.66666666667,71000.0
Amit,IT,82000,1,79666.66666666667,71000.0
Pooja,HR,90000,1,75000.0,71000.0
Vikas,Finance,72000,1,65333.333333333336,71000.0


In [0]:
# 59. Find employees whose salary is:greater than company average salary and belong to the department with the highest average salary



from pyspark.sql.functions import *

# Company average
company_avg_df = emp_df.agg(
    avg("salary").alias("company_avg")
)

# Department averages
dept_avg_df = emp_df.groupBy("dept_id") \
    .agg(avg("salary").alias("dept_avg"))

# Department with highest average salary
top_dept = dept_avg_df.orderBy(
    col("dept_avg").desc()
).limit(1)

# Employees from highest avg department
df = emp_df.join(
    top_dept,
    on="dept_id"
).crossJoin(company_avg_df) \
 .filter(
     col("salary") > col("company_avg")
 ) \
 .select(
     "name",
     "dept_id",
     "salary",
     "dept_avg",
     "company_avg"
 )

df.display()


name,dept_id,salary,dept_avg,company_avg
Rahul,10,75000,79666.66666666667,71000.0
Sneha,10,82000,79666.66666666667,71000.0
Amit,10,82000,79666.66666666667,71000.0


In [0]:
# 61. Find employees whose salary is greater than their department average salary.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id")

dept_avg_df = emp_df.withColumn("d_avg",avg("salary").over(w))\
    .filter(col("salary") > col("d_avg"))\
        .select("name","salary","d_avg")
dept_avg_df.display()        

df_d_avg = emp_df.groupBy("dept_id").agg(avg("salary").alias("dept_avg_salary"))

df = emp_df.join(df_d_avg)\
    .filter(col("salary") > col("dept_avg_salary"))
df.display()    

name,salary,d_avg
Sneha,82000,79666.66666666667
Amit,82000,79666.66666666667
Pooja,90000,75000.0
Vikas,72000,65333.333333333336


emp_id,name,dept_id,salary,dept_id,dept_avg_salary
101,Rahul,10,75000,30,65333.333333333336
101,Rahul,10,75000,40,50000.0
102,Sneha,10,82000,10,79666.66666666667
102,Sneha,10,82000,20,75000.0
102,Sneha,10,82000,30,65333.333333333336
102,Sneha,10,82000,40,50000.0
103,Amit,10,82000,10,79666.66666666667
103,Amit,10,82000,20,75000.0
103,Amit,10,82000,30,65333.333333333336
103,Amit,10,82000,40,50000.0


In [0]:
# 61. pattern dept avg > empsal


from pyspark.sql.functions import *
from pyspark.sql.window import Window

df_d_avg = emp_df.groupBy("dept_id").agg(avg("salary").alias("dept_avg_salary"))

df = emp_df.join(df_d_avg, "dept_id")\
    .filter(col("salary") > col("dept_avg_salary"))
df.display()   

dept_id,emp_id,name,salary,dept_avg_salary
10,102,Sneha,82000,79666.66666666667
10,103,Amit,82000,79666.66666666667
20,106,Pooja,90000,75000.0
30,109,Vikas,72000,65333.333333333336


In [0]:
# 62.Find employees whose salary is greater than:
# 1.department average salary
# 2. AND company average salary

from pyspark.sql.functions import *

d_avg = emp_df.groupBy("dept_id").agg(avg("salary").alias("dept_avg"))
c_avg = emp_df.agg(avg("salary").alias("c_avg"))

emp_sal = emp_df.join(d_avg,"dept_id").crossJoin(c_avg)\
    .filter(((col("salary") > col("dept_avg"))) & (col("salary") > col("c_avg")))\
        .select("name","salary","dept_avg","c_avg")
emp_sal.display()        

name,salary,dept_avg,c_avg
Sneha,82000,79666.66666666667,71000.0
Amit,82000,79666.66666666667,71000.0
Pooja,90000,75000.0,71000.0
Vikas,72000,65333.333333333336,71000.0


In [0]:
 # 63. Find employees whose salary is:less than their department average salary but greater than the company average salary.

 from pyspark.sql.functions import *
 from pyspark.sql.window import Window

 w = Window.partitionBy("dept_id")

 c_avg = emp_df.agg(avg("salary").alias("cp_avg"))

 d_avg = emp_df.groupBy("dept_id").agg(avg("salary").alias("dp_avg"))
 df = emp_df.join(d_avg, "dept_id")\
    .crossJoin(c_avg)\
    .filter((col("salary") < col("dp_avg")) &( col("salary") > col("cp_avg")))\
    .select("name","salary","dp_avg","cp_avg")
 df.display()        

name,salary,dp_avg,cp_avg
Rahul,75000,79666.66666666667,71000.0


In [0]:
# 64. Find employees whose salary is:greater than department average salary ,less than company average salary.

from pyspark.sql.functions import *

dp_Avg = emp_df.groupBy("dept_id").agg(avg("salary").alias("d_avg_sal"))

cp_Avg = emp_df.agg(avg("salary").alias("c_avg_sal"))

df = emp_df.join(dp_Avg,"dept_id").crossJoin(cp_Avg)\
    .filter((col("salary") > col("d_avg_sal")) & (col("salary") < col("c_avg_sal")))\
    .select("name","salary","d_avg_sal","c_avg_sal")
df.display()    

name,salary,d_avg_sal,c_avg_sal


In [0]:
# 65 Find departments where:department average salary < company average  salary  and employee count > 2.

from pyspark.sql.functions import *

d = emp_df.groupBy("dept_id").agg(avg("salary").alias("d_avg"),count("*").alias("emp_count"))
c = emp_df.agg(avg("salary").alias("c_avg"))

df = emp_df.join(d,"dept_id").crossJoin(c)\
    .filter((col("d_avg") < col("c_avg")) &( col("emp_count") > 2))\
    .select("name","d_avg","c_avg","emp_count")
df.display()    

name,d_avg,c_avg,emp_count
Ravi,65333.333333333336,71000.0,3
Anita,65333.333333333336,71000.0,3
Vikas,65333.333333333336,71000.0,3


In [0]:
# 66. Find employees whose salary is:greater than department averag less than department maximum salary

from pyspark.sql.functions import *

d = emp_df.groupBy("dept_id").agg(avg("salary").alias("avg_sal"), max("salary").alias("max_sal"))

df = emp_df.join(d ,"dept_id")\
    .filter((col("salary") > col("avg_sal")) & (col("salary") < col("max_sal")))\
        .select("name","salary","avg_sal","max_sal")
df.display()        

name,salary,avg_sal,max_sal


In [0]:
# 67.Find employees whose salary is:greater than company average salary ,belong to departments having more than 2 employees.
from pyspark.sql.functions import *

c = emp_df.agg(avg("salary").alias("cp_avg"))
d = emp_df.groupBy("dept_id").agg(count("*").alias("emp_count"))

df = emp_df.join(d,"dept_id").crossJoin(c)\
    .filter((col("salary") > col("cp_avg")) & (col("emp_count") > 2))\
        .select("name","salary","cp_avg","emp_count")
df.display()    

name,salary,cp_avg,emp_count
Rahul,75000,71000.0,3
Sneha,82000,71000.0,3
Amit,82000,71000.0,3
Pooja,90000,71000.0,3
Vikas,72000,71000.0,3


In [0]:
# 68. Find departments where:department average salary > company average salary,  department maximum salary > 80000

from pyspark.sql.functions import *

c = emp_df.agg(avg("salary").alias("c_avg"))

d = emp_df.groupBy("dept_id").agg(avg("salary").alias("d_avg"),max("salary").alias("max_sal"))

df = d.crossJoin(c)\
    .filter((col("d_avg") > col("c_avg")) & (col("max_sal") > 80000))\
    .select("dept_id","d_avg","c_avg","max_sal")
df.display()    


dept_id,d_avg,c_avg,max_sal
20,75000.0,71000.0,90000
10,79666.66666666667,71000.0,82000


In [0]:
# 69. Find the running total salary within each department ordered by salary.
from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df.withColumn(
    "running_total",
    sum("salary").over(w)
)

df.show()

+------+-----+-------+------+-------------+
|emp_id| name|dept_id|salary|running_total|
+------+-----+-------+------+-------------+
|   101|Rahul|     10| 75000|        75000|
|   102|Sneha|     10| 82000|       239000|
|   103| Amit|     10| 82000|       239000|
|   104| Neha|     20| 65000|        65000|
|   105|Kiran|     20| 70000|       135000|
|   106|Pooja|     20| 90000|       225000|
|   107| Ravi|     30| 60000|        60000|
|   108|Anita|     30| 64000|       124000|
|   109|Vikas|     30| 72000|       196000|
|   110|Meena|     40| 50000|        50000|
+------+-----+-------+------+-------------+



In [0]:
# 70. Running Average Salary within each department ordered by salary.

from pyspark.sql.functions import avg, col
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id") \
    .orderBy(col("salary")) \
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )

df = emp_df.withColumn(
    "running_avg",
    avg("salary").over(w)
)

df.display()

emp_id,name,dept_id,salary,running_avg
101,Rahul,10,75000,75000.0
103,Amit,10,82000,78500.0
102,Sneha,10,82000,79666.66666666667
104,Neha,20,65000,65000.0
105,Kiran,20,70000,67500.0
106,Pooja,20,90000,75000.0
107,Ravi,30,60000,60000.0
108,Anita,30,64000,62000.0
109,Vikas,30,72000,65333.333333333336
110,Meena,40,50000,50000.0


In [0]:
# 71. Find the running employee count within each department ordered by salary.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))\
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
df.withColumn("running_count", count("*").over(w))

df.display()
    

emp_id,name,dept_id,salary,running_total
101,Rahul,10,75000,75000
102,Sneha,10,82000,157000
103,Amit,10,82000,239000
104,Neha,20,65000,65000
105,Kiran,20,70000,135000
106,Pooja,20,90000,225000
107,Ravi,30,60000,60000
108,Anita,30,64000,124000
109,Vikas,30,72000,196000
110,Meena,40,50000,50000


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df.withColumn(
    "running_avg",
    avg("salary").over(w)
)

df.display()

emp_id,name,dept_id,salary,running_avg
101,Rahul,10,75000,75000.0
103,Amit,10,82000,79666.66666666667
102,Sneha,10,82000,79666.66666666667
104,Neha,20,65000,65000.0
105,Kiran,20,70000,67500.0
106,Pooja,20,90000,75000.0
107,Ravi,30,60000,60000.0
108,Anita,30,64000,62000.0
109,Vikas,30,72000,65333.333333333336
110,Meena,40,50000,50000.0


In [0]:
# 72. Find employees whose running total salary exceeds 150000.

from pyspark.sql.window import Window
from pyspark.sql.functions import *

w = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df.withColumn(
    "running_total",
    sum("salary").over(w)
)\
.filter(col("running_total") > 150000)

df.display()



emp_id,name,dept_id,salary,running_total
103,Amit,10,82000,239000
102,Sneha,10,82000,239000
106,Pooja,20,90000,225000
109,Vikas,30,72000,196000


In [0]:
# 73.Find the first employee in each department where the running total salary exceeds 150000. This adds row_number() on top of the running total pattern. 

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Running total window
w1 = Window.partitionBy("dept_id").orderBy("salary")

# For selecting first matching employee
w2 = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df \
    .withColumn(
        "running_total",
        sum("salary").over(w1)
    ) \
    .filter(col("running_total") > 150000) \
    .withColumn(
        "rn",
        row_number().over(w2)
    ) \
    .filter(col("rn") == 1) \
    .select(
        "name",
        "dept_id",
        "salary",
        "running_total"
    )

df.display()

name,dept_id,salary,running_total
Sneha,10,82000,239000
Pooja,20,90000,225000
Vikas,30,72000,196000


In [0]:
# 74. Find the first employee in each department where: Running Total Salary > 75% of Department Total Salary.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Running total window
w1 = Window.partitionBy("dept_id").orderBy("salary")

# Department total window
w2 = Window.partitionBy("dept_id")

# First matching employee window
w3 = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df \
    .withColumn(
        "running_total",
        sum("salary").over(w1)
    ) \
    .withColumn(
        "dept_total",
        sum("salary").over(w2)
    ) \
    .filter(
        col("running_total") > (col("dept_total") * 0.75)
    ) \
    .withColumn(
        "rn",
        row_number().over(w3)
    ) \
    .filter(col("rn") == 1) \
    .select(
        "name",
        "dept_id",
        "salary",
        "running_total",
        "dept_total"
    )

df.display()


name,dept_id,salary,running_total,dept_total
Amit,10,82000,239000,239000
Pooja,20,90000,225000,225000
Vikas,30,72000,196000,196000
Meena,40,50000,50000,50000


In [0]:
# 76. Find employee tenure in days.
from pyspark.sql.functions import *




df = emp_df.withColumn("tenure", datediff(current_date(),col('hire_date')))

df.display()


emp_id,name,dept_id,salary,hire_date,tenure
101,Rahul,10,75000,null,null
102,Sneha,10,82000,null,null
103,Amit,10,82000,null,null
104,Neha,20,65000,null,null
105,Kiran,20,70000,null,null
106,Pooja,20,90000,null,null
107,Ravi,30,60000,null,null
108,Anita,30,64000,null,null
109,Vikas,30,72000,null,null
110,Meena,40,50000,null,null


In [0]:
# 75. Find employees hired in the last 30 days.

from pyspark.sql.functions import *

df = emp_df \
    .filter(
        datediff(current_date(), col("hire_date")) <= 30)
    ) \
    .select(
        "emp_id",
        "name",
        "hire_date"
    )

df.display()    

emp_id,name,hire_date


In [0]:
# 77. Find employees who completed 1 year in the company.

from pyspark.sql.functions import *
df = emp_df.filter(datediff(current_date(),col('hire_date')) >= 365)\
    .select("name","hire_date")
df.display()    

name,hire_date


In [0]:
# 78. Find employees hired in the current month.

from pyspark.sql.functions import *

df = emp_df\
    .filter(
        (month(col("hire_date")) == month(current_date())) &
        (year(col("hire_date")) == year(current_date()))
    )\
    .select("name","hire_date")
df.display()    


name,hire_date


In [0]:
# 79. Find the number of employees hired each month.

from pyspark.sql import functions as F

df = emp_df.groupBy(month("hire_date").alias("hire_month")).agg(count("*").alias("emp_count")).orderBy(col("hire_month"))

df.display()


hire_month,emp_count
null,10


In [0]:
# 80. Find employees whose work anniversary is in the next 30 days.

from pyspark.sql.functions import *

df = emp_df.withColumn(
    "anniversary_this_year",
    to_date(
        concat(
            year(current_date()),
            lit("-"),
            lpad(month("hire_date"), 2, "0"),
            lit("-"),
            lpad(dayofmonth("hire_date"), 2, "0")
        )
    )
).filter(
    datediff(col("anniversary_this_year"), current_date()).between(0, 30)
).select(
    "name",
    "hire_date",
    "anniversary_this_year"
)

df.display()


name,hire_date,anniversary_this_year


In [0]:
# 81.Find employees whose names start with 'R'.

from pyspark.sql.functions import *

df = emp_df.filter(
    col("name").startswith("R") 
)
df.display()

emp_id,name,dept_id,salary,hire_date
101,Rahul,10,75000,01-03-2023
107,Ravi,30,60000,18-08-2021


In [0]:
# 82. Find employees whose names end with 'a'.

from pyspark.sql.functions import *

df = emp_df.filter(
    col("name").endswith("a")
)

df.display()

emp_id,name,dept_id,salary,hire_date
102,Sneha,10,82000,15-05-2019
104,Neha,20,65000,17-12-2025
106,Pooja,20,90000,11-09-2023
108,Anita,30,64000,12-04-2026
110,Meena,40,50000,11-07-2021


In [0]:
# 83. Find employees whose names contain 'an'

from pyspark.sql.functions import *

df = emp_df.filter(
    col("name").contains("an")
)
df.display()

emp_id,name,dept_id,salary,hire_date
105,Kiran,20,70000,26-04-2022


In [0]:
# 84. Convert all employee names to UPPERCASE.

from pyspark.sql.functions import *

df = emp_df.withColumn("emp_upper", upper(col("name")))
df.display()


emp_id,name,dept_id,salary,hire_date,emp_upper
101,Rahul,10,75000,01-03-2023,RAHUL
102,Sneha,10,82000,15-05-2019,SNEHA
103,Amit,10,82000,07-07-2024,AMIT
104,Neha,20,65000,17-12-2025,NEHA
105,Kiran,20,70000,26-04-2022,KIRAN
106,Pooja,20,90000,11-09-2023,POOJA
107,Ravi,30,60000,18-08-2021,RAVI
108,Anita,30,64000,12-04-2026,ANITA
109,Vikas,30,72000,02-09-2018,VIKAS
110,Meena,40,50000,11-07-2021,MEENA


In [0]:
# 85 . Extract the first 3 characters from employee names.

from pyspark.sql.functions import *

df = emp_df.withColumn("ext",substring(col("name"), 1, 3))
df.display()



emp_id,name,dept_id,salary,hire_date,ext
101,Rahul,10,75000,01-03-2023,Rah
102,Sneha,10,82000,15-05-2019,Sne
103,Amit,10,82000,07-07-2024,Ami
104,Neha,20,65000,17-12-2025,Neh
105,Kiran,20,70000,26-04-2022,Kir
106,Pooja,20,90000,11-09-2023,Poo
107,Ravi,30,60000,18-08-2021,Rav
108,Anita,30,64000,12-04-2026,Ani
109,Vikas,30,72000,02-09-2018,Vik
110,Meena,40,50000,11-07-2021,Mee


In [0]:
# 86.  Extract first 3 characters from employee name.

from pyspark.sql.functions import *

df = emp_df.withColumn("emp_3chr",substring(col("name"),1,3))\
    .select("name")
df.display()

name
Rahul
Sneha
Amit
Neha
Kiran
Pooja
Ravi
Anita
Vikas
Meena


In [0]:
# 87. Split email into username and domain.
from pyspark.sql.functions import *

df = emp_df.withColumn("user",split(col("email"), "@")[0])\
    .withColumn("domain",split(col("email"), "@")[1])\
    .select("email","user","domain")
df.display()    


email,user,domain
rahul@gmail.com,rahul,gmail.com
sneha@gamil.com,sneha,gamil.com
amit123@outlook.com,amit123,outlook.com
neha23@gmail.com,neha23,gmail.com
kiran28@gmail.com,kiran28,gmail.com
pooja65@gmail.com,pooja65,gmail.com
ravi@34gmail.com,ravi,34gmail.com
anita@gmail.com,anita,gmail.com
vikas12@gmail.com,vikas12,gmail.com
meena211@gamil.com,meena211,gamil.com


In [0]:
# 88. convert names to lowercase.

from pyspark.sql.functions import *

df = emp_df.withColumn("l_Name",lower(col("name")))\
    .select("name","l_Name")
df.display()    

name,l_Name
Rahul,rahul
Sneha,sneha
Amit,amit
Neha,neha
Kiran,kiran
Pooja,pooja
Ravi,ravi
Anita,anita
Vikas,vikas
Meena,meena


In [0]:
# 89. Convert names to Proper Case.

from pyspark.sql.functions import *

df = emp_df.withColumn("Name",initcap(col("name")))\
    .select("Name")
df.display()    

Name
Rahul
Sneha
Amit
Neha
Kiran
Pooja
Ravi
Anita
Vikas
Meena


In [0]:
# 90. Combine first_name and last_name.

from pyspark.sql.functions import *

df = emp_df.withColumn("Name",concate_ws(col("first_name"),col("last_name")))\
    .select("Name")
df.display()    

In [0]:
# 91. Replace '-' with '/'.

from pyspark.sql.functions import *

df = emp_df.withColumn("rpl",regexp_replace(col("hire_date"),"-","/"))
df.display()

emp_id,name,dept_id,salary,hire_date,email,rpl
101,Rahul,10,75000,01-03-2023,rahul@gmail.com,01/03/2023
102,Sneha,10,82000,15-05-2019,sneha@gamil.com,15/05/2019
103,Amit,10,82000,07-07-2024,amit123@outlook.com,07/07/2024
104,Neha,20,65000,17-12-2025,neha23@gmail.com,17/12/2025
105,Kiran,20,70000,26-04-2022,kiran28@gmail.com,26/04/2022
106,Pooja,20,90000,11-09-2023,pooja65@gmail.com,11/09/2023
107,Ravi,30,60000,18-08-2021,ravi@34gmail.com,18/08/2021
108,Anita,30,64000,12-04-2026,anita@gmail.com,12/04/2026
109,Vikas,30,72000,02-09-2018,vikas12@gmail.com,02/09/2018
110,Meena,40,50000,11-07-2021,meena211@gamil.com,11/07/2021


In [0]:
# 92. Extract domain from email. 

from pyspark.sql.functions import *

df = emp_df.withColumn("domain", regexp_extract(col("email"), "@(.*)", 1))
df.display()





emp_id,name,dept_id,salary,hire_date,email,domain
101,Rahul,10,75000,01-03-2023,rahul@gmail.com,gmail.com
102,Sneha,10,82000,15-05-2019,sneha@gamil.com,gamil.com
103,Amit,10,82000,07-07-2024,amit123@outlook.com,outlook.com
104,Neha,20,65000,17-12-2025,neha23@gmail.com,gmail.com
105,Kiran,20,70000,26-04-2022,kiran28@gmail.com,gmail.com
106,Pooja,20,90000,11-09-2023,pooja65@gmail.com,gmail.com
107,Ravi,30,60000,18-08-2021,ravi@34gmail.com,34gmail.com
108,Anita,30,64000,12-04-2026,anita@gmail.com,gmail.com
109,Vikas,30,72000,02-09-2018,vikas12@gmail.com,gmail.com
110,Meena,40,50000,11-07-2021,meena211@gamil.com,gamil.com


In [0]:
from pyspark.sql.types import *

data = [
    (101, "Mamta", "Python,SQL,PySpark"),
    (102, "Rahul", "SQL,Power BI"),
    (103, "Sneha", "Python,Azure"),
    (104, "Amit", "PySpark,Databricks,SQL")
]

emp_df = spark.createDataFrame(
    data,
    ["emp_id", "name", "skills"]
)

In [0]:
# 93. Convert skills into separate rows.

from pyspark.sql.functions import *

df = emp_df.select("name",explode(split("Skills", ",")).alias("Skill"), )
df.display()

name,Skill
Mamta,Python
Mamta,SQL
Mamta,PySpark
Rahul,SQL
Rahul,Power BI
Sneha,Python
Sneha,Azure
Amit,PySpark
Amit,Databricks
Amit,SQL


In [0]:
# 94. Find employees having skill "PySpark".

from pyspark.sql.functions import *

df = emp_df.filter(array_contains(split(col("Skills"), ","), "PySpark"))\
    .select("name","Skills")

df.display()

name,Skills
Mamta,"Python,SQL,PySpark"
Amit,"PySpark,Databricks,SQL"


In [0]:
# 95. Count employees per skill. 

from pyspark.sql.functions import *
df = emp_df.select(
    explode(split(col("Skills"), ",")).alias("Skill")
).groupBy("Skill") \
 .agg(count("*").alias("employee_count"))

df.display()




Skill,employee_count
SQL,3
PySpark,2
Python,2
Power BI,1
Azure,1
Databricks,1


In [0]:
# 96. Find unique skills across all employees.

from pyspark.sql.functions import *

df = emp_df.select( explode(split("Skills",",")).alias("skill")).distinct()

df.display()

skill
SQL
PySpark
Python
Power BI
Azure
Databricks


In [0]:
# 97. Collect all employee names into a list.

from pyspark.sql.functions import  *

df = emp_df.select(collect_list(col("name")))


display(df)
    





collect_list(name)
"List(Mamta, Rahul, Sneha, Amit)"


In [0]:
# 98. Collect unique employee names.

from pyspark.sql.functions import *

df = emp_df.select(collect_set(col("name")))
df.display()

collect_set(name)
"List(Rahul, Mamta, Amit, Sneha)"


In [0]:
# 99. Find employees who have either Python or SQL skill.

from pyspark.sql.functions import *
df = emp_df.filter(array_contains(split(col("Skills"), ","), "SQL") | array_contains(split(col("Skills"), ","), "Python"))\
.select("name","Skills")
df.display()



name,Skills
Mamta,"Python,SQL,PySpark"
Rahul,"SQL,Power BI"
Sneha,"Python,Azure"
Amit,"PySpark,Databricks,SQL"


In [0]:
# 100. Count total unique skills.

from pyspark.sql.functions import *

df = emp_df.select(explode(split(col("Skills"), ",")).alias("Skill")).distinct()
print(df.count())

df = emp_df.select(explode(split(col("Skills"), ",")).alias("Skill")).distinct()
count_result = df.count()

df.display()


6


Skill
SQL
PySpark
Python
Power BI
Azure
Databricks


In [0]:
# 101. Find the most popular skill.

from pyspark.sql.functions import *
from pyspark.sql.functions import *

count_result = emp_df.select(
    explode(split(col("Skills"), ",")).alias("Skill")
).distinct().count()

print(count_result)




6


In [0]:
# 
data = [
    ("C101","Jan", "Laptop", 100000),
    ("C102","Jan", "Mouse", 5000),
    ("C103","Feb", "Laptop", 120000),
    ("C104","Feb", "Mouse", 6000),
    ("C105","Mar", "Laptop", 110000),
    ("C106","Mar", "Mouse", 7000)
]

df = spark.createDataFrame(
    data,
    ["cust_id","month", "product", "sales"]
)

df.display()

cust_id,month,product,sales
C101,Jan,Laptop,100000
C102,Jan,Mouse,5000
C103,Feb,Laptop,120000
C104,Feb,Mouse,6000
C105,Mar,Laptop,110000
C106,Mar,Mouse,7000


In [0]:
# 102.   Monthly Sales Report for Each Product? .

from pyspark.sql.functions import *

df.groupBy("product")\
    .pivot("month")\
    .agg(sum("sales"))
df.display()        





cust_id,month,product,sales
C101,Jan,Laptop,100000
C102,Jan,Mouse,5000
C103,Feb,Laptop,120000
C104,Feb,Mouse,6000
C105,Mar,Laptop,110000
C106,Mar,Mouse,7000


In [0]:
# 103. Create a report where:

#Each product appears only once
#Months become columns
#Sales values are shown under each month

from pyspark.sql.functions import *

df.groupBy("cust_id") \
  .pivot("month") \
  .agg(sum("sales"))

df.display()





cust_id,month,product,sales
C101,Jan,Laptop,100000
C102,Jan,Mouse,5000
C103,Feb,Laptop,120000
C104,Feb,Mouse,6000
C105,Mar,Laptop,110000
C106,Mar,Mouse,7000


In [0]:
# 104. HR wants a report showing employee count by department for each location.

data = [
    ("IT", "Pune", 20),
    ("IT", "Mumbai", 15),
    ("IT", "Bangalore", 10),

    ("HR", "Pune", 5),
    ("HR", "Mumbai", 8),
    ("HR", "Bangalore", 3),

    ("Finance", "Pune", 4),
    ("Finance", "Mumbai", 6),
    ("Finance", "Bangalore", 2)
]

df1 = spark.createDataFrame(
    data,
    ["department", "location", "employee_count"]
)

df1.display()
      

department,location,employee_count
IT,Pune,20
IT,Mumbai,15
IT,Bangalore,10
HR,Pune,5
HR,Mumbai,8
HR,Bangalore,3
Finance,Pune,4
Finance,Mumbai,6
Finance,Bangalore,2


In [0]:
# 104. HR wants a report showing employee count by department for each location.
from pyspark.sql.functions import *

df1.groupBy("department")\
    .pivot("location")\
    .agg(sum("employee_count"))\
    .display()

department,Bangalore,Mumbai,Pune
HR,3,8,5
Finance,2,6,4
IT,10,15,20


In [0]:
# Unpivot DataSet :

data = [
    ("Laptop", 100000, 120000, 110000),
    ("Mouse", 5000, 6000, 7000),
    ("Keyboard", 3000, 4000, 5000)
]

df2 = spark.createDataFrame(
    data,
    ["product", "Jan", "Feb", "Mar"]
)

df2.display()

product,Jan,Feb,Mar
Laptop,100000,120000,110000
Mouse,5000,6000,7000
Keyboard,3000,4000,5000


In [0]:
# 105. Convert month columns into rows.

from pyspark.sql.functions import expr

df_unpivot = df2.select(
    "product",
    expr("""
        stack(
            3,
            'Jan', Jan,
            'Feb', Feb,
            'Mar', Mar
        ) as (month, sales)
    """)
)

df_unpivot.display()

product,month,sales
Laptop,Jan,100000
Laptop,Feb,120000
Laptop,Mar,110000
Mouse,Jan,5000
Mouse,Feb,6000
Mouse,Mar,7000
Keyboard,Jan,3000
Keyboard,Feb,4000
Keyboard,Mar,5000


In [0]:
data = [
    ("C101", 2, 1, 0),
    ("C102", 3, 0, 4),
    ("C103", 1, 2, 2)
]

df = spark.createDataFrame(
    data,
    ["customer_id", "Laptop", "Mouse", "Tablet"]
)

df.display()

customer_id,Laptop,Mouse,Tablet
C101,2,1,0
C102,3,0,4
C103,1,2,2


In [0]:
# 106.Customer Purchase Matrix to Transaction Format. 


from pyspark.sql.functions import expr

df.select(
    "customer_id",
    expr("""
        stack(
            3,
            'Laptop', Laptop,
            'Mouse', Mouse,
            'Tablet', Tablet
        ) as (product, quantity)
    """)
).display()

customer_id,product,quantity
C101,Laptop,2
C101,Mouse,1
C101,Tablet,0
C102,Laptop,3
C102,Mouse,0
C102,Tablet,4
C103,Laptop,1
C103,Mouse,2
C103,Tablet,2


In [0]:
# 107. Find the first salary in each department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))
df = emp_df.withColumn("first_salary", first("salary").over(w))

df.display()




emp_id,name,dept_id,salary,hire_date,email,first_salary
101,Rahul,10,75000,01-03-2023,rahul@gmail.com,75000
103,Amit,10,82000,07-07-2024,amit123@outlook.com,75000
102,Sneha,10,82000,15-05-2019,sneha@gamil.com,75000
104,Neha,20,65000,17-12-2025,neha23@gmail.com,65000
105,Kiran,20,70000,26-04-2022,kiran28@gmail.com,65000
106,Pooja,20,90000,11-09-2023,pooja65@gmail.com,65000
107,Ravi,30,60000,18-08-2021,ravi@34gmail.com,60000
108,Anita,30,64000,12-04-2026,anita@gmail.com,60000
109,Vikas,30,72000,02-09-2018,vikas12@gmail.com,60000
110,Meena,40,50000,11-07-2021,meena211@gamil.com,50000


In [0]:
# 108.Find the last salary in each department.

from pyspark.sql.functions import *

from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy(col("salary"))

df = emp_df.withColumn("last_salary",last("salary").over(w))
df.display()

emp_id,name,dept_id,salary,hire_date,email,last_salary
101,Rahul,10,75000,01-03-2023,rahul@gmail.com,75000
103,Amit,10,82000,07-07-2024,amit123@outlook.com,82000
102,Sneha,10,82000,15-05-2019,sneha@gamil.com,82000
104,Neha,20,65000,17-12-2025,neha23@gmail.com,65000
105,Kiran,20,70000,26-04-2022,kiran28@gmail.com,70000
106,Pooja,20,90000,11-09-2023,pooja65@gmail.com,90000
107,Ravi,30,60000,18-08-2021,ravi@34gmail.com,60000
108,Anita,30,64000,12-04-2026,anita@gmail.com,64000
109,Vikas,30,72000,02-09-2018,vikas12@gmail.com,72000
110,Meena,40,50000,11-07-2021,meena211@gamil.com,50000


In [0]:
# 109.Divide employees into 3 salary buckets within each department.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df.withColumn("salary_bucket",ntile(3).over(w))

df.display()


emp_id,name,dept_id,salary,hire_date,email,salary_bucket
101,Rahul,10,75000,01-03-2023,rahul@gmail.com,1
103,Amit,10,82000,07-07-2024,amit123@outlook.com,2
102,Sneha,10,82000,15-05-2019,sneha@gamil.com,3
104,Neha,20,65000,17-12-2025,neha23@gmail.com,1
105,Kiran,20,70000,26-04-2022,kiran28@gmail.com,2
106,Pooja,20,90000,11-09-2023,pooja65@gmail.com,3
107,Ravi,30,60000,18-08-2021,ravi@34gmail.com,1
108,Anita,30,64000,12-04-2026,anita@gmail.com,2
109,Vikas,30,72000,02-09-2018,vikas12@gmail.com,3
110,Meena,40,50000,11-07-2021,meena211@gamil.com,1


In [0]:
# 110. Find the salary percentile rank within each department.

from pyspark.sql.functions import *

from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df.withColumn("salary_percent",percent_rank().over(w))

df.display()

emp_id,name,dept_id,salary,hire_date,email,salary_percent
101,Rahul,10,75000,01-03-2023,rahul@gmail.com,0.0
103,Amit,10,82000,07-07-2024,amit123@outlook.com,0.5
102,Sneha,10,82000,15-05-2019,sneha@gamil.com,0.5
104,Neha,20,65000,17-12-2025,neha23@gmail.com,0.0
105,Kiran,20,70000,26-04-2022,kiran28@gmail.com,0.5
106,Pooja,20,90000,11-09-2023,pooja65@gmail.com,1.0
107,Ravi,30,60000,18-08-2021,ravi@34gmail.com,0.0
108,Anita,30,64000,12-04-2026,anita@gmail.com,0.5
109,Vikas,30,72000,02-09-2018,vikas12@gmail.com,1.0
110,Meena,40,50000,11-07-2021,meena211@gamil.com,0.0


In [0]:
# 111. Find cumulative distribution of salaries within each department.

from pyspark.sql.functions import*
from pyspark.sql.window import Window

w = Window.partitionBy("dept_id").orderBy("salary")

df = emp_df.withColumn("cumlative_distribution_salary", cume_dist().over(w))
df.display()

emp_id,name,dept_id,salary,hire_date,email,cumlative_distribution_salary
101,Rahul,10,75000,01-03-2023,rahul@gmail.com,0.3333333333333333
103,Amit,10,82000,07-07-2024,amit123@outlook.com,1.0
102,Sneha,10,82000,15-05-2019,sneha@gamil.com,1.0
104,Neha,20,65000,17-12-2025,neha23@gmail.com,0.3333333333333333
105,Kiran,20,70000,26-04-2022,kiran28@gmail.com,0.6666666666666666
106,Pooja,20,90000,11-09-2023,pooja65@gmail.com,1.0
107,Ravi,30,60000,18-08-2021,ravi@34gmail.com,0.3333333333333333
108,Anita,30,64000,12-04-2026,anita@gmail.com,0.6666666666666666
109,Vikas,30,72000,02-09-2018,vikas12@gmail.com,1.0
110,Meena,40,50000,11-07-2021,meena211@gamil.com,1.0
